In [17]:
import os
import scanpy as sc
import numpy as np
import plotly.graph_objects as go
from ipywidgets import widgets, Layout

# --- Configuration ---
MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common_synced/"
MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common_synced.h5ad", "halfbrain_yc_2_filtered_common_synced.h5ad",
    "halfbrain_yc_3_filtered_common_synced.h5ad", "halfbrain_yc_4_filtered_common_synced.h5ad",
    "halfbrain_yad_1_filtered_common_synced.h5ad", "halfbrain_yad_2_filtered_common_synced.h5ad",
    "halfbrain_yad_3_filtered_common_synced.h5ad", "halfbrain_yad_4_filtered_common_synced.h5ad",
    "halfbrain_ac_1_filtered_common_synced.h5ad", "halfbrain_ac_2_filtered_common_synced.h5ad",
    "halfbrain_ac_3_filtered_common_synced.h5ad", "halfbrain_ac_4_filtered_common_synced.h5ad",
    "halfbrain_aad_1_filtered_common_synced.h5ad", "halfbrain_aad_2_filtered_common_synced.h5ad",
    "halfbrain_aad_3_filtered_common_synced.h5ad", "halfbrain_aad_4_filtered_common_synced.h5ad"
]
MSI_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4", "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4", "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

sample_dict = dict(zip(MSI_SAMPLE_IDS, MSI_SAMPLE_FILES))

# --- Initialize Figure ---
fig = go.FigureWidget()
fig.update_layout(
    template="plotly_white",
    title="MSI Interactive Spectrum Explorer",
    xaxis_title="m/z",
    yaxis_title="Intensity (10^n)",
    height=650,
    hovermode="x",
    xaxis=dict(
        showgrid=True, 
        gridcolor='lightgrey', 
        # ".4~f" means: up to 4 decimals, but remove trailing zeros.
        tickformat=".4~f",      
        hoverformat=".4~f"
    ),
    yaxis=dict(
        showgrid=True, 
        gridcolor='lightgrey', 
        tickformat=".2e",      # Scientific notation 10^n
        exponentformat="power"
    )
)

def update_spectrum(animal_id, metric, mode, pixel_idx):
    file_path = os.path.join(MSI_INPUT_FOLDER, sample_dict[animal_id])
    
    try:
        adata = sc.read_h5ad(file_path)
    except Exception as e:
        print(f"Error loading {animal_id}: {e}")
        return

    mz_values = adata.var_names.astype(float)
    X = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X
    
    color_map = {"Sum": "royalblue", "Average": "crimson", "Max": "forestgreen"}
    plot_color = color_map.get(metric, "black")

    if mode == "Specific Pixel":
        idx = min(pixel_idx, X.shape[0] - 1)
        intensities = X[idx, :]
        title_suffix = f" (Pixel: {idx})"
        plot_color = "darkviolet"
    else:
        if metric == "Sum": intensities = np.asarray(X.sum(axis=0)).flatten()
        elif metric == "Average": intensities = np.asarray(X.mean(axis=0)).flatten()
        elif metric == "Max": intensities = np.asarray(X.max(axis=0)).flatten()
        title_suffix = f" (Global {metric})"

    # Create Stick data (constant 1.5px width)
    x_sticks, y_sticks = [], []
    for mz, val in zip(mz_values, intensities):
        x_sticks.extend([mz, mz, None])
        y_sticks.extend([0, val, None])

    with fig.batch_update():
        fig.data = []
        
        # Stick Trace
        fig.add_trace(go.Scatter(
            x=x_sticks, 
            y=y_sticks,
            mode='lines',
            line=dict(color=plot_color, width=1.5),
            name=f"{animal_id} {metric}",
            connectgaps=False,
            hoverinfo='skip'
        ))
        
        # Invisible Hover Markers
        fig.add_trace(go.Scatter(
            x=mz_values,
            y=intensities,
            mode='markers',
            marker=dict(color=plot_color, size=1, opacity=0),
            hovertemplate="<b>m/z:</b> %{x:.4~f}<br><b>Int:</b> %{y:.2e}<extra></extra>"
        ))

        fig.update_layout(title=f"MSI Spectrum: {animal_id}{title_suffix}")

# --- UI Layout ---
animal_dropdown = widgets.Dropdown(options=MSI_SAMPLE_IDS, description='Animal:')
metric_dropdown = widgets.Dropdown(options=['Sum', 'Average', 'Max'], description='Metric:')
mode_toggle = widgets.RadioButtons(options=['All Pixels', 'Specific Pixel'], description='Mode:')
pixel_input = widgets.IntText(value=0, description='Pixel Index:', layout=Layout(width='180px'))

ui = widgets.VBox([
    widgets.HBox([animal_dropdown, metric_dropdown]),
    widgets.HBox([mode_toggle, pixel_input])
])

out = widgets.interactive_output(update_spectrum, {
    'animal_id': animal_dropdown,
    'metric': metric_dropdown,
    'mode': mode_toggle,
    'pixel_idx': pixel_input
})

display(ui, fig)

FigureWidget({
    'data': [{'connectgaps': False,
              'hoverinfo': 'skip',
              'line': {'color': 'royalblue', 'width': 1.5},
              'mode': 'lines',
              'name': 'YC_1 Sum',
              'type': 'scatter',
              'uid': '87056816-17c3-4c83-b75a-fe9def527feb',
              'x': [326.1991, 326.1991, None, ..., 1088.8307, 1088.8307, None],
              'y': [0, 2029432.125, None, ..., 0, 2251795.5, None]},
             {'hovertemplate': '<b>m/z:</b> %{x:.4~f}<br><b>Int:</b> %{y:.2e}<extra></extra>',
              'marker': {'color': 'royalblue', 'opacity': 0, 'size': 1},
              'mode': 'markers',
              'type': 'scatter',
              'uid': 'f80dd510-6358-40e5-9385-0c21c14fc02e',
              'x': array([ 326.1991,  337.1936,  339.2312, ..., 1003.7093, 1017.6721, 1088.8307]),
              'y': array([ 2029432.1,  8752830. , 12173503. , ...,  2148389.5,  1963418.1,
                           2251795.5], dtype=float32)}],
    